# LLM Laptop Recommendation Analysis
Analyze persona-specific preferences and GRUM vs Bradley-Terry correlations in the laptop domain.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

ROOT = Path.cwd().parent
sns.set_theme(style="whitegrid", palette="vibrant")

## 1. Load Data

In [ ]:
# TODO: Set your experiment directory
EXP_DIR = ROOT / "results" / "llm" / "llm_laptops-20260326-091953"
output_dir = EXP_DIR / "outputs"

results = []
if output_dir.exists():
    for f in sorted(output_dir.glob("*.json")):
        with open(f, "r") as j:
            results.append(json.load(j))

if results:
    print(f"Loaded {len(results)} trials")

## 2. Intrinsic Preferences (Delta)
Comparing GRUM's extracted social preference to the Bradley-Terry baseline.

In [ ]:
# Laptop names from domain config (could also load from results metadata)
laptop_names = [
    "ASUS VivoBook 15", "Lenovo IdeaPad 3", "HP 15 Laptop", "Dell Inspiron 15",
    "Apple MacBook Air M2", "Lenovo Legion 5", "ASUS ROG Zephyrus G14",
    "Dell XPS 13", "HP Spectre x360", "Lenovo ThinkPad X1", "Apple MacBook Pro 14",
    "ASUS Zenbook 14"
]

grum_deltas = []
bt_deltas = []

for r in results:
    last_step = str(sorted(map(int, r["history"].keys()))[-1])
    entry = r["history"][last_step]
    grum_deltas.append(entry["grum"]["delta"])
    bt_deltas.append(entry["bt"]["delta"])

if grum_deltas:
    df_grum = pd.DataFrame(grum_deltas, columns=laptop_names).melt(var_name="Laptop", value_name="Score")
    order = df_grum.groupby("Laptop")["Score"].mean().sort_values(ascending=False).index
    
    plt.figure(figsize=(12, 8))
    sns.barplot(data=df_grum, y="Laptop", x="Score", order=order, palette="viridis")
    plt.title("GRUM Intrinsic Preferences ($\delta$)")
    plt.show()

## 3. Persona Effects: One-Hot Intersections
We use the learned $B$ matrix and one-hot persona vectors to see how preferences shift for Student, Gamer, etc.

In [ ]:
if results:
    res = results[0]
    last_step = str(sorted(map(int, res["history"].keys()))[-1])
    delta = np.array(res["history"][last_step]["grum"]["delta"])
    B = np.array(res["history"][last_step]["grum"]["interaction"])
    
    personas = ["Student", "Gamer", "Business", "Editor"]
    persona_utilities = {}
    
    for i, name in enumerate(personas):
        # The first 4 features are one-hot persona
        x = np.zeros(B.shape[0])
        x[i] = 1.0
        
        u = delta + x @ B
        persona_utilities[name] = u
        
    df_persona = pd.DataFrame(persona_utilities, index=laptop_names)
    
    plt.figure(figsize=(14, 10))
    sns.heatmap(df_persona, annot=True, cmap="RdYlGn", center=0)
    plt.title("Persona-Specific Predicted Utilities")
    plt.xlabel("Persona")
    plt.ylabel("Laptop")
    plt.show()

## 4. Rank Reversals: Gamer vs Student

In [ ]:
if results:
    df_comp = df_persona[["Student", "Gamer"]].reset_index().rename(columns={"index": "Laptop"})
    df_comp = df_comp.melt(id_vars="Laptop", var_name="Persona", value_name="Utility")
    
    plt.figure(figsize=(12, 6))
    sns.pointplot(data=df_comp, x="Laptop", y="Utility", hue="Persona")
    plt.xticks(rotation=45, ha='right')
    plt.title("Preference Rank Reversals: Student vs Gamer")
    plt.axhline(0, color='black', alpha=0.2)
    plt.show()